# Checking for duplicates in database

This notebook analyses if there are any duplicates inside the tables of database.

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from database.scripts.config import load_config
from database.scripts.connect import connect

# ignore warings
import warnings
warnings.filterwarnings("ignore")

# plot defaults
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# load db config data
config = load_config("database/scripts/database.ini")
# connect to posgres db
conn = connect(config)

# execute sql and return results as DataFrame
def q(sql, params=None):
    return pd.read_sql(sql, conn, params=params)

Connected to the PostgreSQL server.


## Duplicates in table work

Checking for duplicates on entries in doi column.

In [30]:
work_doi = q("""
    SELECT doi, count(*) AS n, array_agg(id) AS ids
    FROM openalex.work
    WHERE doi IS NOT NULL
    GROUP BY doi
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(work_doi)} duplicates found")
work_doi

0 duplicates found


,doi,n,ids


## Handle duplicate DOIs in work table

When inserting works from OpenAlex data duplicates on DOI can appear. To address this issue the one with the newer publication year will be kept.

Insertion Logic:

- If an existing work with same DOI is **newer or equally recent** -> skip the insert.
- If an existing work with same DOI is **older** -> delete it, then insert the new one.
- IF **no existing work** has this DOI -> insert it directly.


Investigations showed that whenever inserting new works a duplicate DOI can occur. The existing work in the database was always the **older** publication. Means, the publication year was less then or equal to the current one.

Within insertion process:
```python
publication_year = get_work_publication_year(doi)
if publication_year is not None and publication_year >= current_publication_year:
    return  # skip if existing work's publication year is newer
if publication_year is not None:
    delete_work(doi) # delete old work
insert_work() # insert current work
```

Checking for duplicates on entries in normalized title column.

In [31]:
work_title = q("""
    SELECT lower(trim(title)) AS title_norm, count(*) AS n, array_agg(id) AS ids
    FROM openalex.work
    GROUP BY title_norm
    HAVING count(*) > 1
    ORDER BY n DESC;
""")
#'''

print(f"{len(work_title)} duplicates found")
work_title

126 duplicates found


,title_norm,n,ids
0,agentic ai-based intelligent study assistant u...,4,"[W7165390545, W7165052073, W7164995226, W71653..."
1,generative ai integration and rag-based enterp...,3,"[W7154620011, W7171159371, W7171257378]"
2,visualseek ai,3,"[W7165137142, W7160655424, W7160603378]"
3,medease : an ai driven medical chatbot using l...,3,"[W7154469721, W7154165199, W7154041098]"
4,aug-wp-175 — sle definitions-api v3: definitiv...,3,"[W7154177220, W7151527246, W7154035720]"
...,...,...,...
121,a data fusion-based framework to integrate mul...,2,"[W4287605364, W3096981884]"
122,aether: adaptive embodied thinking — holistic ...,2,"[W7140326939, W7140302503]"
123,a fully local multi-agent retrieval-augmented ...,2,"[W7160489304, W7160542278]"
124,ai-assisted learning management system using r...,2,"[W7168235388, W7168294195]"


## Duplicates in table author

In [32]:
author = q("""
    SELECT lower(trim(orcid)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.author
    GROUP BY orcid
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(author)} duplicates found")
author

6 duplicates found


,lower,n,ids
0,NaN,6936,"[7987, 7988, 7738, 7727, 7992, 7993, 7994, 799..."
1,https://orcid.org/0000-0001-6825-4697,2,"[15113, 11525]"
2,https://orcid.org/0000-0002-6996-8859,2,"[13705, 13315]"
3,https://orcid.org/0000-0003-2036-0989,2,"[10153, 10151]"
4,https://orcid.org/0000-0003-4313-938x,2,"[597, 13089]"
5,https://orcid.org/0009-0007-6299-2008,2,"[5113, 5114]"


## Duplicates in table institution

In [33]:
institution = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.institution
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(institution)} duplicates found")
institution

24 duplicates found


,lower,n,ids
0,institut de recherche pour le développement,3,"[I1306264927, I4210108561, I4210166444]"
1,arab open university,2,"[I4210139873, I4210090878]"
2,bioinformatics institute,2,"[I4210137637, I4210148498]"
3,carter center,2,"[I1292524976, I4210096273]"
4,cisco systems (united states),2,"[I135428043, I4210129566]"
5,cracow university of technology,2,"[I4210092770, I24881138]"
6,institute of mechanics,2,"[I4210157653, I4210148896]"
7,institute of philosophy,2,"[I4210088627, I4210104258]"
8,institute of physics,2,"[I4210086947, I4210159876]"
9,joint research centre,2,"[I4210162697, I4210118689]"


## Duplicates in table funder

In [34]:
funder = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.funder
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(funder)} duplicates found")
funder

1 duplicates found


,lower,n,ids
0,national science and technology council,2,"[F4320331164, F2461203286]"


## Duplicates in table source

In [35]:
source = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.source
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(source)} duplicates found")
source

12 duplicates found


,lower,n,ids
0,spire - sciences po institutional repository,5,"[S4406922454, S4406922461, S4406922398, S44069..."
1,geodesy and cartography,2,"[S12104227, S4210169005]"
2,infoscience (ecole polytechnique fédérale de l...,2,"[S4306400487, S4306400488]"
3,international journal of computer science and ...,2,"[S154051528, S4390963318]"
4,international journal of educational development,2,"[S5407050336, S20152851]"
5,journal of geophysical research atmospheres,2,"[S207178839, S4210205282]"
6,london school of economics and political scien...,2,"[S4306401594, S4306401593]"
7,pub – publications at bielefeld university (bi...,2,"[S4306401671, S4306401670]"
8,quaestiones geographicae,2,"[S157707066, S4210171875]"
9,repository@nottingham (university of nottingham),2,"[S4306402481, S4306402483]"


## Duplicates in table locations

In [36]:
locations = q("""
    SELECT lower(trim(pdf_url)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.locations
    GROUP BY pdf_url
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(locations)} duplicates found")
locations

1 duplicates found


,lower,n,ids
0,None,4013,"[oai:mdpi.com:/2073-8994/15/5/1020/, 41646097,..."


## Duplicates in table keyword

In [37]:
keyword = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.keyword
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(keyword)} duplicates found")
keyword

0 duplicates found


,lower,n,ids


## Duplicates in table domain

In [38]:
domain = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.domain
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(domain)} duplicates found")
domain

0 duplicates found


,lower,n,ids


## Duplicates in table field

In [39]:
field = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.field
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(field)} duplicates found")
field

0 duplicates found


,lower,n,ids


## Duplicates in table subfield

In [40]:
subfield = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.subfield
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(subfield)} duplicates found")
subfield

4 duplicates found


,lower,n,ids
0,industrial and manufacturing engineering,2,"[2209, 2311]"
1,genetics,2,"[1311, 2716]"
2,pharmacology,2,"[2736, 3004]"
3,neurology,2,"[2808, 2728]"


## Duplicates in table topic

In [41]:
topic = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.topic
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{len(topic)} duplicates found")
topic

0 duplicates found


,lower,n,ids
